In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Install All Required Libraries
# ═══════════════════════════════════════════════════════════════
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-core
!pip install -q faiss-gpu sentence-transformers
!pip install -q datasets
!pip install -q fastapi uvicorn pyngrok
!pip install -q pydantic

print("All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 46.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requir

In [3]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Load Fine-Tuned Model & Tokenizer
# ═══════════════════════════════════════════════════════════════

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "KareemAboalnoor/faqeeh-qwen2.5-7b-egyptian-legal-v2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Very important for Qwen
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model (this takes ~2-3 minutes)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

# Small optimization for inference
torch.set_grad_enabled(False)

print("=" * 55)
print(f"Model Loaded Successfully")
print(f"Device : {model.device}")
print(f"Parameters : {model.num_parameters() / 1e9:.1f}B")
print("=" * 55)

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model Loaded Successfully
Device : cuda:0
Parameters : 7.6B


In [4]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Build RAG Vector Store (FAISS + Egyptian Legal Corpus)
# ═══════════════════════════════════════════════════════════════

from datasets import load_dataset
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

# ──────────────────────────────────────────────────────────────
# 3.1 Load Egyptian Legal Corpus
# ──────────────────────────────────────────────────────────────
print("Loading Egyptian Legal Corpus...")

corpus_ds = load_dataset(
    "dataflare/egypt-legal-corpus",
    split="train"
)

# ──────────────────────────────────────────────────────────────
# 3.2 Convert to LangChain Documents
# ──────────────────────────────────────────────────────────────
print("Creating legal documents...")

documents = []

for row in corpus_ds:

    text = row.get("text", "").strip()

    if len(text) < 50:
        continue

    documents.append(
        Document(
            page_content=text,
            metadata={
                "law_name": row.get("law_name", "Unknown"),
                "categories": row.get("categories", [])
            }
        )
    )

print(f"Loaded {len(documents)} legal documents")

# ──────────────────────────────────────────────────────────────
# 3.3 Split into Chunks
# ──────────────────────────────────────────────────────────────
print("Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=[
        "\n\n",
        "\n",
        ".",
        " ",
        ""
    ]
)

docs_chunks = text_splitter.split_documents(documents)

# Remove duplicate chunks
print("Removing duplicate chunks...")

seen = set()
unique_chunks = []

for doc in docs_chunks:

    key = doc.page_content.strip()

    if key in seen:
        continue

    seen.add(key)
    unique_chunks.append(doc)

docs_chunks = unique_chunks

print(f"Total unique chunks: {len(docs_chunks)}")

# ──────────────────────────────────────────────────────────────
# 3.4 Load Embedding Model
# ──────────────────────────────────────────────────────────────
print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "batch_size": 32,
        "normalize_embeddings": True
    }
)

# ──────────────────────────────────────────────────────────────
# 3.5 Build FAISS
# ──────────────────────────────────────────────────────────────
print("Building FAISS Vector Store...")

vectorstore = FAISS.from_documents(
    docs_chunks,
    embeddings
)

# ──────────────────────────────────────────────────────────────
# 3.6 Retriever
# ──────────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 10
    }
)

# ──────────────────────────────────────────────────────────────
# 3.7 Load Reranker
# ──────────────────────────────────────────────────────────────
print("Loading reranker...")

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

print("═" * 60)
print("RAG Pipeline Ready")
print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(docs_chunks)}")
print("Retriever : Top-10 Similarity Search")
print("Embedding : BAAI/bge-m3")
print("Reranker  : BAAI/bge-reranker-v2-m3")
print("═" * 60)

/tmp/ipykernel_58/885219883.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading Egyptian Legal Corpus...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2434 [00:00<?, ? examples/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

════════════════════════════════════════════════════════════
RAG Pipeline Ready
Documents : 2434
Chunks    : 54686
Retriever : Top-10 Similarity Search
Embedding : BAAI/bge-m3
Reranker  : BAAI/bge-reranker-v2-m3
════════════════════════════════════════════════════════════


In [5]:
# ═══════════════════════════════════════════════════════════════
# Cell 4 : Guardrails + Memory + Helper Functions
# ═══════════════════════════════════════════════════════════════

import re
import torch

# ==============================================================
# Helper Generation Function
# ==============================================================

def generate_direct(prompt_text: str, max_new_tokens=80):

    messages = [
        {
            "role": "system",
            "content":
            """أنت مساعد قانوني متخصص في القانون المصري.

أجب باللغة العربية فقط.

إذا كان السؤال خارج القانون المصري فأجب بكلمة:
خارج التخصص

يمنع استخدام أي لغة أخرى.
"""
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return response.strip()


# ==============================================================
# Legal Keywords
# ==============================================================

LEGAL_TRIGGERS = {

    "قانون","مادة","محكمة","محامي","قضية","دعوى",
    "حكم","عقوبة","حبس","غرامة","شرطة","نيابة",
    "سرقة","تزوير","نصب","احتيال","رشوة",
    "إيجار","بيع","شراء","ملكية","أرض","شقة",
    "عقد","فسخ","تعويض","حق","حقوق","التزام",
    "ورث","ميراث","تركة",
    "طلاق","خلع","نفقة","حضانة","زواج",
    "عمل","عامل","موظف","فصل","راتب",
    "شيك","إيصال أمانة",
    "محضر","القسم","بلاغ",
    "استئناف","نقض","تظلم",
    "ضريبة","توثيق","شهر عقاري",

    "اتنصب عليا",
    "ضربني",
    "هددني",
    "رفعت قضية",
    "اتفصلت",
    "حقوقي",
    "حقي"

}


BLOCKED_TOPICS = {

    "طبخ","وصفة","كيكة","بيتزا",
    "دواء","مرض","علاج","طبيب",
    "برمجة","python","java","html","css",
    "ذكاء اصطناعي","chatgpt","claude","gemini",
    "مباراة","كورة","صلاح","ميسي",
    "فيلم","مسلسل","اغنية",
    "رياضيات","فيزياء","كيمياء",
    "سفر","فندق","طيران"

}

REJECTION_RESPONSE = """
أنا مساعد قانوني متخصص في القانون المصري فقط.

يمكنني مساعدتك في:

• قانون العمل
• قانون الأسرة
• العقوبات
• العقارات
• العقود
• القضايا والإجراءات القضائية
"""


# ==============================================================
# Greetings
# ==============================================================

GREETING_PATTERNS = [

    r"^(أهلا|اهلا|مرحبا|السلام عليكم|هاي|هلو|hello|hi)[\s!.،]*$",

    r"^(شكرا|شكراً|thanks|thank you)[\s!.،]*$",

    r"^(من انت|مين انت|who are you)[\s!.؟]*$",

    r"^(عامل ايه|كيف حالك|how are you)[\s!.؟]*$"

]

GREETING_RESPONSES = {

    "greeting":
    "أهلاً بك، أنا فقيه، مساعد قانوني متخصص في القانون المصري.",

    "thanks":
    "العفو، يسعدني مساعدتك في أي استفسار قانوني.",

    "who":
    "أنا فقيه، مساعد ذكاء اصطناعي متخصص في القانون المصري.",

    "how":
    "بخير، شكراً لك. كيف يمكنني مساعدتك قانونياً؟"

}


# ==============================================================
# Question Classification
# ==============================================================

def classify_question(question):

    q = question.strip().lower()

    for pattern in GREETING_PATTERNS:

        if re.match(pattern, q, re.IGNORECASE):

            if "شكر" in q or "thank" in q:
                return "greeting", GREETING_RESPONSES["thanks"]

            elif "مين" in q or "who" in q:
                return "greeting", GREETING_RESPONSES["who"]

            elif "عامل" in q or "كيف" in q:
                return "greeting", GREETING_RESPONSES["how"]

            return "greeting", GREETING_RESPONSES["greeting"]

    for word in LEGAL_TRIGGERS:

        if word in q:
            return "legal", None

    for word in BLOCKED_TOPICS:

        if word in q:
            return "blocked", REJECTION_RESPONSE

    if len(re.findall(r'[\u4e00-\u9fff]', q)) > 2:
        return "blocked", REJECTION_RESPONSE

    if len(re.findall(r'[\u0400-\u04ff]', q)) > 2:
        return "blocked", REJECTION_RESPONSE

    if len(q) < 4:
        return "unknown", "من فضلك وضح سؤالك القانوني."

    return "legal", None


def is_legal_question(question):

    category, response = classify_question(question)

    if category == "legal":
        return True, None

    return False, response


# ==============================================================
# Output Validation
# ==============================================================

def validate_model_output(response):

    if len(re.findall(r'[\u4e00-\u9fff]', response)) > 5:
        return REJECTION_RESPONSE

    if len(re.findall(r'[\u0400-\u04ff]', response)) > 5:
        return REJECTION_RESPONSE

    english = len(re.findall(r"[A-Za-z]", response))

    if english > 40:
        return REJECTION_RESPONSE

    arabic = len(re.findall(r'[\u0600-\u06FF]', response))

    if arabic < 20:
        return REJECTION_RESPONSE

    return response


# ==============================================================
# Conversation Memory
# ==============================================================

class ConversationMemory:

    def __init__(self, max_turns=5):

        self.max_turns = max_turns
        self.history = []

    def add(self, question, answer):

        self.history.append({

            "user": question,
            "assistant": answer

        })

        if len(self.history) > self.max_turns:
            self.history.pop(0)

    def clear(self):
        self.history = []

    def has_history(self):
        return len(self.history) > 0

    def get_history_text(self):

        if not self.history:
            return ""

        text = ""

        for i, item in enumerate(self.history, 1):

            text += f"[رسالة {i}]\n"

            text += f"المستخدم: {item['user']}\n"

            text += f"فقيه: {item['assistant']}\n\n"

        return text.strip()


memory = ConversationMemory(max_turns=5)

# ==============================================================
# Ambiguity Checker
# ==============================================================

def is_ambiguous(question: str):

    prompt = f"""
حدد هل السؤال التالي واضح قانونياً أم لا.

إذا كان واضحاً أجب فقط:

واضح

إذا كان غير واضح اطلب من المستخدم سؤالاً واحداً فقط للتوضيح.

السؤال:

{question}
"""

    result = generate_direct(
        prompt,
        max_new_tokens=40
    ).strip()

    if "واضح" in result:
        return None

    return result


# ==============================================================
# Query Reformulation
# ==============================================================

def reformulate_query(question: str):

    history = memory.get_history_text()

    if history and len(question.split()) <= 15:

        prompt = f"""
أنت خبير في القانون المصري.

اعتمد على المحادثة السابقة ثم حوّل السؤال الجديد إلى
جملة بحث قانونية واحدة مناسبة للبحث داخل قاعدة بيانات القوانين.

لا تجب على السؤال.

المحادثة:

{history}

السؤال:

{question}

جملة البحث:
"""

    else:

        prompt = f"""
حوّل السؤال التالي من العامية المصرية إلى
صيغة عربية فصحى مناسبة للبحث داخل قاعدة بيانات قانونية.

لا تجب على السؤال.

السؤال:

{question}

جملة البحث:
"""

    result = generate_direct(
        prompt,
        max_new_tokens=50
    )

    result = (
        result
        .replace('"',"")
        .replace(":", "")
        .strip()
    )

    if len(result) < 5:
        return question

    return result


# ==============================================================
# SYSTEM PROMPT
# ==============================================================

SYSTEM_PROMPT = """
أنت "فقيه"

مساعد قانوني متخصص في القانون المصري فقط.

=========================================
قواعد أساسية
=========================================

1- أجب باللغة العربية فقط.

2- لا تستخدم أي لغة أخرى.

3- إذا كان السؤال خارج القانون المصري فاعتذر وارفض الإجابة.

4- لا تخترع إطلاقاً:

- مواد قانونية
- أرقام قوانين
- أحكام محاكم
- أسماء قوانين
- نصوص تشريعية

إذا لم تكن متأكداً فقل:

"لا أملك سنداً قانونياً مؤكداً لهذه المعلومة."

=========================================
طريقة الإجابة
=========================================

أولاً اقرأ النصوص القانونية المسترجعة.

إذا كانت النصوص تجيب على السؤال بشكل مباشر:

- اعتمد عليها.
- اذكر المادة.
- اذكر اسم القانون.
- استشهد بها.

إذا كانت النصوص المسترجعة غير مرتبطة بالسؤال أو ضعيفة:

يمكنك الاعتماد على المعرفة التي اكتسبتها أثناء Fine-Tuning بشرط:

- عدم اختلاق أي مادة.
- عدم اختلاق رقم قانون.
- عدم اختلاق أي سند قانوني.

إذا لم تكن متأكداً فاذكر أن الإجابة مبنية على المعرفة القانونية العامة.

=========================================
أسلوب الكتابة
=========================================

اجعل الإجابة:

- واضحة
- احترافية
- منظمة
- طويلة نسبياً
- سهلة الفهم

استخدم العناوين التالية عند توفر المعلومات:

## الإجابة

## التحليل القانوني

## السند القانوني

## رقم المادة

## الإجراءات المقترحة

## ملاحظات

=========================================
النصوص القانونية المسترجعة

{context}
"""

print("=" * 60)
print("Guardrails Loaded Successfully")
print("=" * 60)

Guardrails Loaded Successfully


In [6]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Complete Generation Pipeline
# ═══════════════════════════════════════════════════════════════

from transformers import TextIteratorStreamer
from threading import Thread
import torch

# ═══════════════════════════════════════════════════════════════
# 5.1: DOCUMENT DEDUPLICATION
# ═══════════════════════════════════════════════════════════════

def filter_and_deduplicate_docs(docs: list) -> list:
    """
    Remove duplicate and very short chunks.
    """
    seen = set()
    filtered = []

    for doc in docs:
        content = doc.page_content.strip()

        if len(content) < 100:
            continue

        fingerprint = content[:250]

        if fingerprint in seen:
            continue

        seen.add(fingerprint)
        filtered.append(doc)

    return filtered


# ═══════════════════════════════════════════════════════════════
# 5.2: BUILD CONTEXT
# ═══════════════════════════════════════════════════════════════

def build_context(docs: list) -> str:

    if not docs:
        return ""

    context = []

    for i, doc in enumerate(docs, start=1):

        law = doc.metadata.get("law_name", "Unknown")

        context.append(
            f"""
========== Document {i} ==========
Law Name:
{law}

Legal Text:
{doc.page_content.strip()}
"""
        )

    return "\n\n".join(context)


# ═══════════════════════════════════════════════════════════════
# 5.3: RERANKING
# ═══════════════════════════════════════════════════════════════

def retrieve_best_documents(query):

    raw_docs = retriever.invoke(query)

    filtered_docs = filter_and_deduplicate_docs(raw_docs)

    if len(filtered_docs) == 0:
        return [], False, []

    pairs = [
        [query, doc.page_content]
        for doc in filtered_docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(filtered_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    best_docs = [doc for doc, score in ranked[:3]]

    best_scores = [float(score) for doc, score in ranked[:3]]

    confidence = False

    if len(best_scores):

        if best_scores[0] >= 0.25:
            confidence = True

    return best_docs, confidence, best_scores

# ═══════════════════════════════════════════════════════════════
# 5.4: MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════

def process_question(question: str) -> dict:

    # -------------------------------------------------------
    # 1. Apply guardrails
    # -------------------------------------------------------
    is_legal, rejection_msg = is_legal_question(question)

    if not is_legal:
        return {
            "rejected": True,
            "rejection_message": rejection_msg,
        }

    # -------------------------------------------------------
    # 2. Check for ambiguity
    # -------------------------------------------------------
    ambiguity = is_ambiguous(question)

    if ambiguity:
        return {
            "rejected": True,
            "rejection_message": ambiguity,
        }

    # -------------------------------------------------------
    # 3. Reformulate the query
    # -------------------------------------------------------
    search_query = reformulate_query(question)

    # -------------------------------------------------------
    # 4. Retrieve relevant documents
    # -------------------------------------------------------
    best_docs, rag_confident, scores = retrieve_best_documents(search_query)

    # -------------------------------------------------------

    # 5. Decide whether to use retrieved context
    # -------------------------------------------------------

    if len(best_docs) > 0:

        context = build_context(best_docs)

        source = "RAG + Fine-Tuning"

    else:

        context = ""

        source = "Fine-Tuning"

    # -------------------------------------------------------
    # 6. Build the prompt
    # -------------------------------------------------------
    system_message = SYSTEM_PROMPT.format(
        context=context
    )

    messages = [
        {
            "role": "system",
            "content": system_message,
        }
    ]

    # Add conversation history
    if memory.has_history():

        for turn in memory.history:

            messages.append(
                {
                    "role": "user",
                    "content": turn["user"],
                }
            )

            messages.append(
                {
                    "role": "assistant",
                    "content": turn["assistant"],
                }
            )

    messages.append(
        {
            "role": "user",
            "content": question,
        }
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    # -------------------------------------------------------
    # Return the processed inputs
    # -------------------------------------------------------
    return {

        "rejected": False,

        "prompt_inputs": inputs,

        "source": source,

        "rag_confident": rag_confident,

        "scores": scores,

        "context_info":
        f"Source: {source} | "
        f"Retrieved Docs: {len(best_docs)} | "
        f"Scores: {[round(x,3) for x in scores]}"
    }

# ═══════════════════════════════════════════════════════════════
# 5.5: GENERATION CONFIG
# ═══════════════════════════════════════════════════════════════

GENERATION_CONFIG = {

    # Maximum response length
    "max_new_tokens": 1024,

    # Sampling parameters
    "do_sample": True,

    "temperature": 0.15,

    "top_p": 0.90,

    "top_k": 40,

    # Reduce repetition
    "repetition_penalty": 1.15,


    # Proper termination tokens
    "pad_token_id": tokenizer.eos_token_id,

    "eos_token_id": tokenizer.eos_token_id,
}

# ═══════════════════════════════════════════════════════════════
# 5.6: QUICK GENERATE
# ═══════════════════════════════════════════════════════════════

def quick_generate(question: str):

    result = process_question(question)

    if result["rejected"]:

        memory.add(
            question,
            result["rejection_message"]
        )

        return result["rejection_message"]

    print("=" * 60)
    print(result["context_info"])
    print("=" * 60)

    with torch.no_grad():

        outputs = model.generate(

            **result["prompt_inputs"],

            **GENERATION_CONFIG,

        )

    input_length = result["prompt_inputs"]["input_ids"].shape[1]

    response = tokenizer.decode(

        outputs[0][input_length:],

        skip_special_tokens=True,

    ).strip()

    response = validate_model_output(response)

    memory.add(question, response)

    return response

In [9]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Comprehensive Tests
# ═══════════════════════════════════════════════════════════════

def run_test(title, question):
    print("\n" + "═" * 70)
    print(title)
    print("═" * 70)
    print(f"Question: {question}\n")
    answer = quick_generate(question)
    print(answer)


# Clear conversation memory
memory.clear()

# Test 1
run_test(
    "TEST 1 - Legal Question (Egyptian Dialect)",
    "لو حد نصب عليا اعمل ايه؟"
)

# Test 2 (Follow-up)
run_test(
    "TEST 2 - Follow-up Question",
    "طب اشرحلي الإجراءات بالتفصيل"
)

# Test 3
memory.clear()
run_test(
    "TEST 3 - Out-of-Domain Question",
    "ما هي مكونات الكيكه؟"
)

# Test 4
memory.clear()
run_test(
    "TEST 4 - Greeting",
    "السلام عليكم"
)

# Test 5
memory.clear()
run_test(
    "TEST 5 - Legal Question (Modern Standard Arabic)",
    "ما هي حقوق العامل في حالة الفصل التعسفي وفقاً لقانون العمل المصري؟"
)

# Test 6
memory.clear()
run_test(
    "TEST 7 - Non-Legal Question",
    "من هو محمد صلاح؟"
)

print("\n All Tests Finished.")


══════════════════════════════════════════════════════════════════════
TEST 1 - Legal Question (Egyptian Dialect)
══════════════════════════════════════════════════════════════════════
Question: لو حد نصب عليا اعمل ايه؟

Source: RAG + Fine-Tuning | Retrieved Docs: 3 | Scores: [0.001, 0.0, 0.0]
في الأحوال دي، لو حد نصب عليك، اتبع الخطوات التانية:

1. حافظ على أدلة الدليل الجنائي: كالمكالمات والرسائل الإلكترونية والصور والأدلة الأخرى المتعلقة بالنصب.

2. اتصل بالشرطة فورًا وقم بتقديم بلاغ رسمي ضد المتهم.

3. اطلب من الشرطة إعداد محضر رسمى يتضمن جميع التفاصيل.

4. قد تحتاج إلى تقديم شكوى أمام النيابة العامة لإصدار أمر ضبط وإحضار للمتهم.

5. إذا تم القبض عليه، يمكنك المشاركة في التحقيق معه وتوفير كل المعلومات اللازمة.

تذكر إن الوقت مهم جداً في مثل هذه الحالات، لذا اتخذ الإجراءات فورًا.

══════════════════════════════════════════════════════════════════════
TEST 2 - Follow-up Question
══════════════════════════════════════════════════════════════════════
Question: طب اشرحلي الإجراءات بالت

In [8]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: FastAPI Streaming Server + ngrok
# ═══════════════════════════════════════════════════════════════

import uvicorn
import threading
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel as PydanticBase
from pyngrok import ngrok
from transformers import TextIteratorStreamer
from threading import Thread

app = FastAPI(title="فقيه - المساعد القانوني المصري")


class QuestionRequest(PydanticBase):
    question: str


@app.post("/reset")
async def reset_memory():
    memory.clear()
    return {"status": "memory_cleared"}


@app.post("/ask_stream")
async def ask_stream(req: QuestionRequest):

    result = process_question(req.question)

    if result["rejected"]:

        def rejection_stream():
            yield result["rejection_message"]
            memory.add(req.question, result["rejection_message"])

        return StreamingResponse(
            rejection_stream(),
            media_type="text/plain; charset=utf-8",
        )

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = {
        **result["prompt_inputs"],
        **GENERATION_CONFIG,
        "streamer": streamer,
        "pad_token_id": tokenizer.eos_token_id,
    }

    thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs,
    )
    thread.start()

    def generate_chunks():

        full_response = ""

        for new_text in streamer:
            full_response += new_text
            yield new_text

        final_response = validate_model_output(full_response)

        memory.add(req.question, final_response)

    return StreamingResponse(
        generate_chunks(),
        media_type="text/plain; charset=utf-8",
    )


# ============================================================
# Insert your ngrok authentication token here
# ============================================================

NGROK_TOKEN = "3Gm6wzye5DrlvqoOf4r0Eq8zich_3UP6mVEr2Z7SXUh7jn7Ye"

ngrok.set_auth_token(NGROK_TOKEN)

try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(8000).public_url

print("=" * 60)
print(f"Public URL : {public_url}")
print(f"API Docs   : {public_url}/docs")
print("=" * 60)


def run_server():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="error",
    )
    server = uvicorn.Server(config)
    server.run()


server_thread = threading.Thread(
    target=run_server,
    daemon=True,
)
server_thread.start()

Public URL : https://coherent-endowment-outpost.ngrok-free.dev
API Docs   : https://coherent-endowment-outpost.ngrok-free.dev/docs
